# EpiCMITHypo

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
4. [Load features](#Load-features)
5. [Load weights into base model](#Load-weights-into-base-model)
6. [Load reference values](#Load-reference-values)
7. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
8. [Check all clock parameters](#Check-all-clock-parameters)
9. [Normal feature ranges](#Normal-feature-ranges)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.EpiCMITHypo)

class EpiCMITHypo(epiTOC1):
    pass



In [3]:
model = pya.models.EpiCMITHypo()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "epicmithypo"
model.metadata["data_type"] = "DNA methylation"  # Paper: methylation
model.metadata["species"] = "Homo sapiens"  # Paper: Homo sapiens
model.metadata["year"] = 2020
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Duran-Ferrer, M., et al. \"The proliferative history shapes the DNA methylome of B-cell tumors and predicts clinical outcome.\" Nature Cancer 1 (2020): 1066-1081."
model.metadata["doi"] = "https://doi.org/10.1038/s43018-020-00131-2"
model.metadata["notes"] = "Hypomethylation component of epiCMIT: a 1,164-CpG score ranging from 0 to 1 that tracks low-to-high relative proliferative history in normal and neoplastic B cells."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["B cells"]  # Paper: normal and neoplastic B cells
model.metadata["predicts"] = ["mitotic age"]  # Paper: The author tutorial states both component clocks range from 0 to 1 and report relative proliferative history.
model.metadata["training_target"] = ["replicative history"]  # Paper: relative proliferative history
model.metadata["unit"] = ["proportion"]  # Paper: The author tutorial states both component clocks range from 0 to 1 and report relative proliferative history.
model.metadata["model_type"] = "complement of mean methylation"  # Paper: Complement of mean methylation score
model.metadata["platform"] = ["Illumina 450K", "Illumina EPIC"]  # Paper: Illumina 450K; Illumina EPIC
model.metadata["population"] = "human, age unspecified"  # Paper: 1,595 human samples spanning normal B-cell subpopulations and 14 B-cell tumor subtypes
model.metadata["journal"] = "Nature Cancer"
model.metadata["last_author"] = "José I. Martín-Subero"
model.metadata["n_features"] = 1164
model.metadata["citations"] = 104
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

In [5]:
supplementary_url = "https://static-content.springer.com/esm/art%3A10.1038%2Fs43018-020-00131-2/MediaObjects/43018_2020_131_MOESM3_ESM.xlsx"
supplementary_file_name = "epicmit.xlsx"
os.system(f"curl -sL -o {supplementary_file_name} {supplementary_url}")

0

## Load features

In [6]:
df = pd.read_excel('epicmit.xlsx', sheet_name='Table 23')
df = df[df['epiCMIT.class'].astype(str).str.contains('hypo', case=False)]
model.features = df['Name'].tolist()

## Load weights into base model

In [7]:
weights = torch.tensor([-1.0]).unsqueeze(0)
intercept = torch.tensor([1.0])

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = [-1]*len(model.features)

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = "mean"
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Duran-Ferrer, Marti, et al. "The proliferative history shapes '
             'the DNA methylome of B-cell tumors and predicts clinical '
             'outcome." Nature Cancer 1.11 (2020): 1066-1081.',
 'clock_name': 'epicmithypo',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1038/s43018-020-00131-2',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2020}
reference_values: [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1]... [Total elements: 1164]
preprocess_name: 'mean'
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg21870274', 'cg17810211', 'cg08263140', 'cg05663262', 'cg00055603', 'cg05042706', 'cg25469314', 'cg16515477', 'cg07910726'

## Normal feature ranges

In [ ]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

## Basic test

In [ ]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: epicmit.xlsx
